# Weather Data Analysis

Notebook phân tích thống kê dữ liệu thời tiết Hà Nội.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/processed/Hanoi_final.csv").exists() and (PROJECT_ROOT.parent / "data/processed/Hanoi_final.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Time: {df['time'].min()} -> {df['time'].max()}")
df.head()

## 1. Chất lượng dữ liệu

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(4),
    "unique": df.nunique(dropna=False),
}).sort_values("missing", ascending=False)

quality_summary.head(40)

In [ ]:
time_diff = df["time"].diff().dropna()
pd.DataFrame({
    "metric": ["duplicate_timestamps", "most_common_interval", "non_hourly_intervals"],
    "value": [df["time"].duplicated().sum(), time_diff.mode().iloc[0], (time_diff != pd.Timedelta(hours=1)).sum()],
})

## 2. Thống kê mô tả

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns
df[numeric_columns].describe().T.round(3)

## 3. Tổng hợp theo giờ và tháng

In [ ]:
hourly_summary = df.groupby("hour").agg(
    temperature_mean=("temperature_2m", "mean"),
    humidity_mean=("relative_humidity_2m", "mean"),
    rain_mean=("rain", "mean"),
    wind_speed_mean=("wind_speed_10m", "mean"),
).round(3)

hourly_summary

In [ ]:
monthly_summary = df.groupby("month").agg(
    temperature_mean=("temperature_2m", "mean"),
    temperature_min=("temperature_2m", "min"),
    temperature_max=("temperature_2m", "max"),
    humidity_mean=("relative_humidity_2m", "mean"),
    rain_sum=("rain", "sum"),
    wind_speed_mean=("wind_speed_10m", "mean"),
).round(3)

monthly_summary

## 4. Tương quan với nhiệt độ

In [ ]:
correlation_with_target = (
    df[numeric_columns]
    .corr()["temperature_2m"]
    .sort_values(key=lambda series: series.abs(), ascending=False)
)

correlation_with_target.head(25)

## 5. Outlier theo IQR

In [ ]:
outlier_rows = []
for column in ["temperature_2m", "relative_humidity_2m", "rain", "wind_speed_10m", "pressure_msl"]:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[column] < lower) | (df[column] > upper)).sum()
    outlier_rows.append({"feature": column, "lower": lower, "upper": upper, "outliers": count})

pd.DataFrame(outlier_rows).round(3)